<a href="https://colab.research.google.com/github/shahedtheboss/MLflow-wine-classification-project/blob/main/wine_quality_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Install the ultra-fast Rust package manager
!pip install uv

# 2. Use uv to install MLflow directly into the system environment
!uv pip install --system mlflow dagshub


Using Python 3.13.18 environment at: /usr
Checked 2 packages in 83ms


In [3]:
import os
import warnings
import sys

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error,mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
from urllib.parse import urlparse
import mlflow
from mlflow.models.signature import infer_signature
import mlflow.sklearn
import dagshub
import logging



In [5]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [9]:
logging.basicConfig(level=logging.WARN)
logger = logging.getLogger(__name__)




In [10]:
def eval_metric(actual,pred):
  rmse = np.sqrt(mean_squared_error(actual,pred))
  mae = mean_absolute_error(actual,pred)
  r2 = r2_score(actual,pred)
  return rmse,mae,r2

In [12]:
import os

if __name__ == "__main__":
  warnings.filterwarnings("ignore")
  np.random.seed(42)

  # Download latest version
  download_path = kagglehub.dataset_download("uciml/red-wine-quality-cortez-et-al-2009")
  print("Path to dataset directory:", download_path)

  try:
    # Find the CSV file inside the directory
    files = os.listdir(download_path)
    csv_files = [f for f in files if f.endswith('.csv')]

    if not csv_files:
      raise FileNotFoundError("No CSV files found in the downloaded directory.")

    # Use the first CSV file found
    file_path = os.path.join(download_path, csv_files[0])
    print(f"Loading CSV file from: {file_path}")

    # Note: This dataset actually uses a comma "," as a separator rather than a semicolon ";"
    data = pd.read_csv(file_path)
    display(data.head())
  except Exception as e:
    logger.exception(
      "Unable to load training & test CSV. Error: %s", e
    )

Using Colab cache for faster access to the 'red-wine-quality-cortez-et-al-2009' dataset.
Path to dataset directory: /kaggle/input/red-wine-quality-cortez-et-al-2009
Loading CSV file from: /kaggle/input/red-wine-quality-cortez-et-al-2009/winequality-red.csv


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [ ]:
train, test = train_test_split(data)

train_x = train.drop(["quality"])